# Discount levels and buying: the half-buy discount

**Probit Regression** · a worked example from the StatNex teaching guide

**The story.** A store tested seven discount levels on small groups of shoppers and counted how many bought the product at each level.

**The question.** How big a discount does it take before half of the shoppers buy?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/probit_regression.csv` (7 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Discount (%),Shoppers shown,Shoppers who bought
5.0,10.0,1.0
10.0,12.0,2.0
15.0,10.0,3.0
20.0,11.0,5.0
25.0,12.0,8.0
30.0,10.0,8.0
35.0,10.0,10.0

These cells save `collinear_columns.py`, `hosmer_lemeshow.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile collinear_columns.py
"""Which explanatory variables are exact linear combinations of the ones before them.

Such a variable adds no information and makes the model impossible to estimate. Each
column is centred and scaled to length 1, then regressed on the columns already kept;
if nothing is left over (squared residual length at most 1e-10) the column is
redundant. A constant column is redundant too. The *later* member of a redundant pair
is the one reported, so a variable listed first survives.
"""

import numpy as np


def collinear_columns(values, names, tolerance=1e-10):
    centred = values - values.mean(axis=0)
    scale = np.linalg.norm(centred, axis=0)
    scale[scale == 0] = 1.0
    centred = centred / scale
    dropped, kept = [], []
    for j, name in enumerate(names):
        column = centred[:, j]
        if kept:
            basis = centred[:, kept]
            column = column - basis @ np.linalg.lstsq(basis, column, rcond=None)[0]
        if column @ column <= tolerance or values[:, j].max() == values[:, j].min():
            dropped.append(name)
        else:
            kept.append(j)
    return dropped

In [ ]:
%%writefile hosmer_lemeshow.py
"""Hosmer and Lemeshow's goodness-of-fit test, with the app's grouping.

The cases are put in order of predicted probability and cut into ten groups, and each
group's observed events are set against the sum of its predicted probabilities
(Hosmer, Lemeshow and Sturdivant 2013, section 5.2.2).

The textbook leaves ties open, so the rule is this: group k ends at the predicted
probability of the case at position ceil(k N / 10) in that order, and holds every case
above the previous group's end and at or below its own. Cases with equal predicted
probabilities are therefore never split, and ties can leave fewer than ten groups; the
degrees of freedom are the groups actually formed less 2. A row of grouped data counts
as `trials` cases and is never split either. It is Stata's `estat gof, group(10)`.
"""

import numpy as np
import pandas as pd
from scipy import stats


def hosmer_lemeshow(events, trials, fitted, groups=10):
    """The groups as a table, and the chi-square, its df and its P."""
    events, trials, fitted = (np.asarray(a, dtype=float) for a in (events, trials, fitted))
    order = np.argsort(fitted, kind="stable")
    count = np.cumsum(trials[order])
    total = count[-1]
    # The first case whose running count reaches k N / groups -- compared as
    # k N <= groups x count, so that no fraction is rounded on the way.
    ends = np.unique([fitted[order][np.argmax(count * groups >= k * total)]
                      for k in range(1, groups + 1)])
    group = np.searchsorted(ends, fitted) + 1
    table = pd.DataFrame({"group": group, "n": trials, "observed": events,
                          "expected": trials * fitted}).groupby("group").sum()
    table["upper end"] = ends
    chi2 = ((table["observed"] - table["expected"]) ** 2
            / (table["expected"] * (1 - table["expected"] / table["n"]))).sum()
    df = len(table) - 2
    return table, chi2, df, stats.chi2.sf(chi2, df)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from collinear_columns import collinear_columns
from hosmer_lemeshow import hosmer_lemeshow

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Shoppers who bought"
predictors = ["Discount (%)"]
max_iter = 50
confidence = 95 / 100
# Each row is a group: how many responded, out of how many subjects.
trials_column = "Shoppers shown"
frame = data[[response, *predictors, trials_column]].apply(pd.to_numeric, errors="coerce")
complete = frame.notna().all(axis=1)
used = frame[complete]
events = used[response].to_numpy(float)
trials = used[trials_column].to_numpy(float)
rate = events / trials
# Every subject is a case, so n counts subjects rather than rows.
n = int(trials.sum())
summary = pd.DataFrame({"level": used[predictors[0]].to_numpy(float), "responders": events,
                        "subjects": trials, "rate": rate})
summary.loc["total"] = [np.nan, events.sum(), trials.sum(), events.sum() / trials.sum()]
print(summary)
print("rows left out as incomplete:", (~complete).sum())

## Variables that are linear combinations of others

In [ ]:
values = used[predictors].to_numpy(float)
dropped = collinear_columns(values, predictors)
print("\nLinearly dependent:", dropped or "none")
# They are left out of the model.
predictors = [name for name in predictors if name not in dropped]
values = used[predictors].to_numpy(float)

## The probit model

In [ ]:
# P(response) = Phi(x'b), Phi being the standard normal distribution function. The rate
# is fitted with the subjects as frequency weights, which is the binomial likelihood.
# The constant is the last column, as the app prints it.
design = np.column_stack([values, np.ones(len(rate))])
names = [*predictors, "Constant"]
probit = sm.families.Binomial(sm.families.links.Probit())
model = sm.GLM(rate, design, family=probit, freq_weights=trials).fit(maxiter=max_iter, tol=1e-12)
null = sm.GLM(rate, np.ones(len(rate)), family=probit, freq_weights=trials).fit(
    maxiter=max_iter, tol=1e-12)


# -2 sum[r log p + (n - r) log(1 - p)], written without the binomial coefficient. C(n, r)
# is constant given the data, so it cancels from every test, but it shifts -2 log L
# itself; the app leaves it out, and so does this.
def minus_two_log_likelihood(p):
    successes = rate * trials
    return -2 * (successes * np.log(p) + (trials - successes) * np.log(1 - p)).sum()


deviance = minus_two_log_likelihood(model.fittedvalues)
null_deviance = minus_two_log_likelihood(null.fittedvalues)

# Pseudo R²: Cox-Snell, Nagelkerke and McFadden.
cox_snell = 1 - np.exp(-(null_deviance - deviance) / n)
accuracy = pd.DataFrame([[cox_snell, cox_snell / (1 - np.exp(-null_deviance / n)),
                          1 - deviance / null_deviance]],
                        columns=["Cox-Snell", "Nagelkerke", "McFadden"])
chi2 = null_deviance - deviance
df = len(predictors)
model_tests = pd.DataFrame(
    [[null_deviance, null_deviance + 2, np.nan, np.nan, np.nan],
     [deviance, deviance + 2 * design.shape[1], chi2, df, stats.chi2.sf(chi2, df)]],
    index=["without the variables", "with the variables"],
    columns=["-2LL", "AIC", "chi-square", "df", "P"])
print()
print(accuracy.to_string(index=False))
print()
print(model_tests)

## Coefficients

In [ ]:
# A coefficient is how far one unit of x moves the response probability's probit
# (its standard-normal score). Wald test (b/SE)² on 1 df.
z = stats.norm.ppf(0.5 + confidence / 2)
b, se = model.params, model.bse
coefficients = pd.DataFrame({"b": b, "SE": se, "lower": b - z * se, "upper": b + z * se,
                             "Wald": (b / se) ** 2, "df": 1, "P": stats.chi2.sf((b / se) ** 2, 1)},
                            index=names)
print()
print(coefficients)

## Hosmer-Lemeshow goodness of fit

In [ ]:
# Ten groups by predicted probability (deciles); a row's subjects, and cases with equal
# probabilities, stay in one group, so ties can leave fewer (hosmer_lemeshow.py).
# df = groups - 2.
responders = events
if responders is not None:
    hl_groups, hl_chi2, hl_df, hl_p = hosmer_lemeshow(responders, trials, model.fittedvalues)
    hosmer_lemeshow_test = [hl_chi2, hl_df, hl_p, len(hl_groups)]
    print()
    print(hl_groups)
    print(f"Hosmer-Lemeshow chi-square {hl_chi2:.4f}, df {hl_df}, P {hl_p:.4f}")

## What the result says

Probit regression fits an S-shaped curve to a share that rises with a dose. Here the dose is the discount, and the share is the shoppers who bought at each of seven discount levels.

The discount clearly matters: its coefficient is 0.10 for each percentage point, with a P value of almost zero (chance alone would hardly give such a link), and the model test gives a chi-square of 32.2. The Hosmer-Lemeshow test compares the observed counts with the fitted ones. Its P value of 0.95 shows no sign of a poor fit.

In the data, 5 of 11 shoppers bought at a 20% discount and 8 of 12 at 25%. The fitted curve reaches one half where the coefficient times the discount cancels the constant of -1.99, which is at a discount of about 20%, because 0.10 times 20 is about 2. So it takes a discount of about 20% before half the shoppers buy. That is not automatically the most profitable discount.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.